In [4]:
import sys
from pathlib import Path
from torchvision import transforms,datasets
from torch.utils.data import Subset,DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score,precision_score,recall_score
sys.path.append(str(Path("..")))
from src.dataset import train_data,val_loader,train_dataset,test_path,train_loader
from src.sampler import BalancedBatchSampler

In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [6]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x


num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []

for epoch in range(num_epochs):


    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        targets = torch.zeros(
            labels.size(0),
            num_classes,
            device=device
        )
        
        targets.scatter_(
            1,
            labels.unsqueeze(1),
            1
        )
        loss = criterion(outputs, targets)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total

    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            
            targets = torch.zeros(
                labels.size(0),
                num_classes,
                device=device
            )

            targets.scatter_(1,labels.unsqueeze(1),1)

            loss = criterion(outputs, targets)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average= None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    val_f1_scores.append(val_f1)
    val_f1_scores_by_class .append(f1_score_by_class)



    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )
    
    print('F1_Score by each class')
    print(f1_score_by_class)
    
    
    print('=======================================================================================')

Epoch [1/100] | Train Loss: 0.3056 | Train Acc: 0.4730 | Val Loss: 0.2027 | Val Acc: 0.6742 | Val Macro F1: 0.6177
F1_Score by each class
[0.64670659 0.77386935 0.56521739 0.60377358 0.57142857 0.77272727
 0.83157895 0.17647059]
Epoch [2/100] | Train Loss: 0.1544 | Train Acc: 0.7646 | Val Loss: 0.1503 | Val Acc: 0.7718 | Val Macro F1: 0.7486
F1_Score by each class
[0.79470199 0.87958115 0.5323741  0.64227642 0.84662577 0.82587065
 0.90909091 0.55813953]
Epoch [3/100] | Train Loss: 0.0825 | Train Acc: 0.8870 | Val Loss: 0.1295 | Val Acc: 0.8348 | Val Macro F1: 0.8193
F1_Score by each class
[0.8        0.88235294 0.70175439 0.74468085 0.84810127 0.90291262
 0.96907216 0.70588235]
Epoch [4/100] | Train Loss: 0.0403 | Train Acc: 0.9606 | Val Loss: 0.1516 | Val Acc: 0.8213 | Val Macro F1: 0.8055
F1_Score by each class
[0.77922078 0.92307692 0.68627451 0.71657754 0.83443709 0.88780488
 0.97409326 0.64285714]
Epoch [5/100] | Train Loss: 0.0203 | Train Acc: 0.9820 | Val Loss: 0.1683 | Val Acc:

In [7]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.7730
Test Macro F1: 0.7681
Test F1 by class:
[0.77419355 0.91836735 0.63063063 0.60869565 0.81904762 0.82142857
 0.95918367 0.61333333]
